# 🏥 P13 — Hospital Readmission Prediction (Diabetes)
**Author:** Mohamed · M3 · ML Engine Portfolio · Project 13

**Dataset:** Diabetes 130-US Hospitals · UCI Repository · 101,763 patient encounters · 1999–2008

**Date:** August 2026

---

## 🎯 Project Objectives
1. Load and profile 101,763 diabetic patient encounters from 130 US hospitals
2. Handle special null marker `?` and high-missing columns (weight 97%, payer_code 40%)
3. Convert age ranges `[70-80)` to numeric midpoints
4. Engineer clinical readmission risk features: high_utilization, long_stay, uses_insulin
5. Export clean dataset ready for Streamlit ML Engine app

**Regression Target:** `time_in_hospital` — days in hospital (1–14)

**Classification Target:** `readmitted_30` — 1 if readmitted within 30 days (11.2%)

---

## ⚠️ Key Notes
- Null marker: `?` — use `na_values='?'` on load
- 11.2% readmission rate → `class_weight='balanced'` MANDATORY
- 101K rows → `CalibratedClassifierCV(LinearSVC)` for SVM only
- `max_glu_serum` / `A1Cresult` filled with `'None'` = test not ordered
- `keep_default_na=False` when reading clean CSV to preserve 'None' strings

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
CLR = {'primary':'#1565c0','success':'#2e7d32','danger':'#c62828',
       'purple':'#6a1b9a','amber':'#f57f17','grey':'#546e7a'}
print('✅ Imports loaded')

---
## 📂 Section 2 — Data Loading & Profile

In [ ]:
# KEY: na_values='?' — this dataset uses ? as null marker
df = pd.read_csv('diabetic_data.csv', na_values='?', low_memory=False)
print(f'Shape: {df.shape}')
print(f'\nNull breakdown:')
null_cols = df.isnull().sum().sort_values(ascending=False)
for col, n in null_cols[null_cols>0].items():
    print(f'  {col:30s}: {n:,} ({n/len(df)*100:.1f}%)')
print(f'\nTarget distribution:')
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).round(3))

---
## 🛠 Section 4 — Data Cleaning

In [ ]:
# Fill before drop
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult']     = df['A1Cresult'].fillna('None')
df['race']          = df['race'].fillna('Unknown')

# Drop high-null + identifier columns
drop_cols = ['encounter_id','patient_nbr','weight',
             'payer_code','medical_specialty',
             'diag_1','diag_2','diag_3']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# Remove invalid gender
df = df[df['gender'] != 'Unknown/Invalid']
print(f'✅ Clean shape: {df.shape}')
print(f'✅ Total nulls: {df.isnull().sum().sum()}')

---
## ⚙️ Section 5 — Feature Engineering

In [ ]:
# Age midpoint encoding
age_map = {'[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,
           '[40-50)':45,'[50-60)':55,'[60-70)':65,'[70-80)':75,
           '[80-90)':85,'[90-100)':95}
df['age_mid'] = df['age'].map(age_map)
print(f'✅ age_mid: mean={df["age_mid"].mean():.1f} years')

# Binary classification target
df['readmitted_30']  = (df['readmitted'] == '<30').astype(int)
df['readmitted_any'] = (df['readmitted'] != 'NO').astype(int)
print(f'✅ readmitted_30: {df["readmitted_30"].mean()*100:.1f}% positive')

In [ ]:
# Medication encoding
med_map = {'No':0,'Steady':1,'Down':2,'Up':3}
med_cols = ['metformin','repaglinide','nateglinide','chlorpropamide',
            'glimepiride','acetohexamide','glipizide','glyburide',
            'tolbutamide','pioglitazone','rosiglitazone','acarbose',
            'miglitol','troglitazone','tolazamide','examide',
            'citoglipton','insulin','glyburide-metformin',
            'glipizide-metformin','glimepiride-pioglitazone',
            'metformin-rosiglitazone','metformin-pioglitazone']
for col in med_cols:
    if col in df.columns:
        df[col+'_enc'] = df[col].map(med_map).fillna(0).astype(int)
print('✅ 23 medication columns encoded (No=0, Steady=1, Down=2, Up=3)')

In [ ]:
# Categorical encoding
le = LabelEncoder()
for col in ['race','gender','max_glu_serum','A1Cresult','change','diabetesMed']:
    if col in df.columns:
        df[col+'_enc'] = le.fit_transform(df[col].astype(str))

# Clinical risk features
df['total_visits']    = df['number_outpatient']+df['number_emergency']+df['number_inpatient']
df['total_med_changes']= df[[c for c in df.columns if c.endswith('_enc') and
                               any(m in c for m in med_cols)]]\
                           .apply(lambda r: (r==2).sum()+(r==3).sum(), axis=1)
df['num_active_meds'] = df[[c for c in df.columns if c.endswith('_enc') and
                              any(m in c for m in med_cols)]]\
                           .apply(lambda r: (r>=1).sum(), axis=1)

q75_visits = df['total_visits'].quantile(0.75)
df['high_utilization']   = (df['total_visits'] > q75_visits).astype(int)
df['long_stay']          = (df['time_in_hospital'] >= 7).astype(int)
df['uses_insulin']       = (df['insulin_enc'] >= 1).astype(int)
df['is_senior']          = (df['age_mid'] >= 65).astype(int)
df['emergency_admission']= (df['admission_type_id'] == 1).astype(int)

print('✅ Clinical risk features created:')
for feat, label in [('high_utilization','High Utilization'),
                    ('long_stay','Long Stay'),
                    ('uses_insulin','Uses Insulin'),
                    ('is_senior','Senior')]:
    rate = df.groupby(feat)['readmitted_30'].mean()*100
    print(f'   {label}: {rate.get(0,0):.2f}% vs {rate.get(1,0):.2f}% readmit')

---
## 📊 Section 6 — Key Analysis

In [ ]:
# Readmission by utilization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utilization vs readmission
hu = df.groupby('high_utilization')['readmitted_30'].mean()*100
bars = axes[0].bar(['Normal Utilization','High Utilization'],
                   hu.values, color=[CLR['success'],CLR['danger']], edgecolor='white')
axes[0].axhline(df['readmitted_30'].mean()*100, color=CLR['primary'],
                lw=2, ls='--', label=f'Avg {df["readmitted_30"].mean()*100:.1f}%')
axes[0].set_ylabel('Readmission Rate %')
axes[0].set_title('Readmission Rate: High vs Normal Utilization', fontweight='bold')
axes[0].legend()
for bar, val in zip(bars, hu.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

# Prior inpatient visits vs readmission
inp_rate = df.groupby('number_inpatient')['readmitted_30'].mean()*100
inp_rate = inp_rate[inp_rate.index<=10]
axes[1].bar(inp_rate.index, inp_rate.values, color=CLR['purple'], edgecolor='white')
axes[1].axhline(df['readmitted_30'].mean()*100, color=CLR['danger'],
                lw=2, ls='--', label=f'Avg {df["readmitted_30"].mean()*100:.1f}%')
axes[1].set_xlabel('Prior Inpatient Visits')
axes[1].set_ylabel('Readmission Rate %')
axes[1].set_title('Readmission Rate by Prior Inpatient Visits', fontweight='bold')
axes[1].legend()
plt.suptitle('Hospital Utilization → Readmission Risk', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'→ High utilization: {hu.get(1,0):.2f}% vs {hu.get(0,0):.2f}% — {hu.get(1,0)/hu.get(0,1):.1f}× higher')

In [ ]:
# A/B Test: High utilization
gA = df[df['high_utilization']==1]['readmitted_30'].astype(float)
gB = df[df['high_utilization']==0]['readmitted_30'].astype(float)
t_stat, p_val = stats.ttest_ind(gA, gB, equal_var=False)
pooled   = np.sqrt((gA.std()**2 + gB.std()**2) / 2)
cohens_d = (gA.mean() - gB.mean()) / (pooled + 1e-10)

print('=== A/B TEST: HIGH vs NORMAL UTILIZATION — READMISSION ===')
print(f'  High utilization  : {gA.mean()*100:.2f}%  n={len(gA):,}')
print(f'  Normal utilization: {gB.mean()*100:.2f}%  n={len(gB):,}')
print(f'  t-stat  : {t_stat:.4f}')
print(f'  p-value : {p_val:.6f}')
print(f'  Cohen d : {cohens_d:.4f}')
print(f'  Significant: {"YES ✅" if p_val < 0.05 else "NO"}')

In [ ]:
# Business KPIs
COST_PER_READMISSION = 15000
COST_INTERVENTION    = 500

total_readmit = df['readmitted_30'].sum()
total_cost    = total_readmit * COST_PER_READMISSION
prevented     = int(total_readmit * 0.30)
int_cost      = total_readmit * COST_INTERVENTION
savings       = prevented * COST_PER_READMISSION - int_cost

print('=== BUSINESS KPIs ===')
print(f'  Total 30-day readmissions : {total_readmit:,}')
print(f'  Estimated annual cost     : ${total_cost/1e6:.1f}M')
print(f'  Intervention cost         : ${int_cost/1e6:.1f}M')
print(f'  Potential savings (30%)   : ${savings/1e6:.1f}M')
print(f'  ROI                       : {savings/int_cost*100:,.0f}%')

---
## 💾 Section 7 — Final Validation & Export

In [ ]:
print('='*55)
print('  P13 HOSPITAL READMISSION — FINAL SUMMARY')
print('='*55)
print(f'  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Total nulls    : {df.isnull().sum().sum()}')
print(f'  Memory         : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()
print(f'  REG target     : time_in_hospital')
print(f'    Mean         : {df["time_in_hospital"].mean():.2f} days')
print(f'    Range        : 1–14 days')
print()
print(f'  CLF target     : readmitted_30')
print(f'    Positive     : {df["readmitted_30"].sum():,} ({df["readmitted_30"].mean()*100:.1f}%)')
print(f'    Negative     : {(df["readmitted_30"]==0).sum():,}')
print('    → class_weight="balanced" MANDATORY')
print()
print('  Top risk factors:')
print(f'    High utilization: {df[df["high_utilization"]==1]["readmitted_30"].mean()*100:.2f}% vs {df[df["high_utilization"]==0]["readmitted_30"].mean()*100:.2f}%')
print(f'    Senior patients : {df[df["is_senior"]==1]["readmitted_30"].mean()*100:.2f}% vs {df[df["is_senior"]==0]["readmitted_30"].mean()*100:.2f}%')
print('='*55)

In [ ]:
df.to_csv('readmission_clean.csv', sep=',', decimal='.', index=False, encoding='utf-8-sig')

# KEY: keep_default_na=False prevents 'None' strings being read back as NaN
df_check = pd.read_csv('readmission_clean.csv', keep_default_na=False)
print(f'✅ Saved: readmission_clean.csv')
print(f'   Shape: {df_check.shape}')
print(f'   Nulls: {df_check.isnull().sum().sum()}')
print()
print('📌 Copy readmission_clean.csv to Repo_13/data/')
print('⚠️  Use File Explorer COPY — do NOT open in Excel!')

---
## 📝 Section 8 — Key Notes

### Data Cleaning Decisions
- `?` null marker → `na_values='?'` on load
- `weight` (97%), `medical_specialty` (49%), `payer_code` (40%) → dropped
- `max_glu_serum` + `A1Cresult` → filled with `'None'` (test not ordered — informative)
- `age` ranges → midpoint numeric for ML
- `keep_default_na=False` when re-reading CSV to preserve 'None' strings

### Target Variables
- **Regression:** `time_in_hospital` — moderate R² expected (many confounders)
- **Classification:** `readmitted_30` — **11.2% positive rate — SEVERE IMBALANCE**
  → `class_weight='balanced'` MANDATORY · evaluate F1/Recall/AUC only

### Top Clinical Predictors
1. **high_utilization** — 18.93% vs 9.72% · **strongest feature** ★
2. **number_inpatient** — each prior admission adds measurable risk
3. **long_stay** — 13.23% vs 10.62%
4. **uses_insulin** — 12.14% vs 10.04%
5. **is_senior** — 11.61% vs 10.23%

---
*Mohamed · M3 · ML Engine Portfolio · P13 Hospital Readmission*